# Which levers raise nacelle output? A line simulation to try

**Part of the *Robotics for Offshore Wind* Virtual Research Building (AICOR, University of Bremen), prepared for the Siemens Energy Innovation Lab, Cuxhaven.**

This notebook runs the discrete-event model of a nacelle assembly line from the research twin
[`CuxhavenSiteTwin`](https://github.com/beetz-m/CuxhavenSiteTwin), pinned to the commit the building reads its numbers from.
You can switch robot and design levers on and off and see what they do to output, and how sure the model is.

**Read the results with these limits in mind.**
- The twin is **reconstructed from public and second-hand sources**. It is not the real plant, and it uses no Siemens Energy data.
- **Station times are our own estimates** (labelled ILLUSTRATIVE in the twin's parameter register). Treat the numbers as a model's answers to "what if", not as forecasts.
- **Each run simulates 60 working days.** We repeat it with 8 random seeds and report the mean, the spread and how often each station was the bottleneck.

Run the cells from top to bottom (*Run → Run All Cells*). It takes about 15 seconds.

In [ ]:
import os, sys, json, statistics, math, collections, subprocess
TWIN = os.path.join(os.environ.get("REPO_DIR", ".."), "CuxhavenSiteTwin")
if not os.path.isdir(TWIN):                      # local run next to a checkout
    TWIN = os.path.abspath(os.path.join("..", "CuxhavenSiteTwin"))
sys.path.insert(0, os.path.join(TWIN, "sim"))
import line_des as D
commit = subprocess.run(["git", "-C", TWIN, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
proc, legs, veh = D.load_inputs()
print("twin commit", commit, "| seeds", D.SEEDS)

## 1. Calibrate the baseline

The model has a fill parameter. It stands for the share of each station's takt spent on work the model does not
itemise (cabling details, hoses, inspection, documentation). The twin sets it by aiming at an output target of
294 nacelles a year, a figure received second-hand and not public. The next cell shows whether that target can be
reached at all.

In [ ]:
fill = D.calibrate(proc, legs, veh)
cal = dict(D.CALIBRATION)
print(json.dumps(cal, indent=1))
print(f"\nTarget {cal['target_per_day']*250:.0f}/yr; itemised work alone gives {cal['rate_without_floor_per_day']*250:.0f}/yr; "
      f"target reachable: {cal['target_reachable']}; fill pinned down by the target: {cal['identifiable']}")

## 2. A helper that runs one case with all seeds

In [ ]:
def run_case(sc):
    reps = [D.simulate(dict(sc, seed=s), proc, legs, veh) for s in D.SEEDS]
    per_year = [r["per_working_day"] * 250 for r in reps]
    m = round(statistics.mean(r["per_working_day"] for r in reps), 3) * 250   # same rounding as line_des.run()
    ci = 1.96 * statistics.stdev(per_year) / math.sqrt(len(per_year))
    bn = collections.Counter(r["bottleneck"] for r in reps)
    return dict(name=sc["name"], mean=m, ci95=ci, lo=min(per_year), hi=max(per_year), bottlenecks=dict(bn),
                lead_h=statistics.median(r["lead_h_median"] for r in reps))

def show(res):
    print(f"{'case':62s} {'per year':>9s} {'±95%':>6s} {'range':>11s}  bottleneck (seeds)")
    for r in res:
        bn = ", ".join(f"{k}×{v}" for k, v in sorted(r["bottlenecks"].items(), key=lambda kv: -kv[1]))
        print(f"{r['name'][:62]:62s} {r['mean']:9.0f} {r['ci95']:6.1f} {r['lo']:5.0f}–{r['hi']:<5.0f}  {bn}")

## 3. Robot levers against design levers

These are the twin's own scenarios:
- **S4** adds robot nutrunners and 16-head tensioning, a change to bolting only.
- **S10** introduces plug-in pre-terminated skids and pre-made looms, a design change that cuts B6 cabling by 40 %.
- **S13** adds the other content reductions.
- **NG1** adds a B6 robot cell with mobile manipulators.
- **NG2** adds mobile bolting.
- **NG3** replaces 15 % of the allowance with itemised work models. This is an assumption about modelling, not a robot.

In [ ]:
pick = ["S0", "S4", "S10", "S13", "NG1", "NG2", "NG3"]
scs = {s["name"].split()[0]: s for s in D.scenarios(fill)}
res = [run_case(scs[k]) for k in pick]
show(res)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 3.6))
y = range(len(res))
ax.barh(list(y), [r["mean"] for r in res], xerr=[r["ci95"] for r in res], color="#0B3D6B", alpha=.85, capsize=3)
ax.set_yticks(list(y)); ax.set_yticklabels([r["name"][:58] for r in res], fontsize=8); ax.invert_yaxis()
ax.axvline(res[0]["mean"], color="#C8102E", lw=1, ls="--"); ax.set_xlabel("nacelles per year (250 working days), mean of 8 seeds ± 95 %")
ax.set_title(f"Line model, twin {commit} (illustrative station times)", fontsize=10); plt.tight_layout(); plt.show()

**What to look for.**
- **Robot bolting (S4) and the B6 robot cell (NG1 after S13)** barely move output. In this model they do not touch the station that limits the line. Their case would rest on safety, ergonomics and documented quality, not throughput.
- **The design change (S10)** moves output, because it shortens the work at the station that is the bottleneck.
- **NG3** jumps only because we assume part of the allowance goes away.
- **Bottleneck counts:** when the count is split across stations, the model cannot say which station limits the line. That is a reason to measure station times in the plant before investing.

## 4. Try your own case

Change the levers and press **Run**. Without widgets, edit the dictionary in the last cell instead.

In [ ]:
try:
    import ipywidgets as W
    from IPython.display import display
    w = dict(
        work_h_day=W.Dropdown(options=[("3 shifts", 21.0), ("2 shifts", 14.0)], value=21.0, description="Shifts"),
        a450=W.IntSlider(value=2, min=1, max=4, description="450 t AGVs"),
        buffers=W.IntSlider(value=0, min=0, max=2, description="Buffers"),
        robot_bolting=W.Checkbox(value=False, description="Robot nutrunners"),
        tension_speedup=W.FloatSlider(value=1.0, min=1.0, max=3.0, step=0.5, description="Tensioning ×"),
        b6_outfit_factor=W.FloatSlider(value=1.0, min=0.3, max=1.0, step=0.05, description="B6 cabling ×"),
        fill_factor=W.FloatSlider(value=1.0, min=0.6, max=1.1, step=0.05, description="Allowance ×"),
        test_h=W.Dropdown(options=[("as modelled", None), ("8 h", 8.0)], value=None, description="End test"),
    )
    out = W.Output(); btn = W.Button(description="Run", button_style="primary")
    def go(_):
        kw = {k: v.value for k, v in w.items()}
        f = kw.pop("fill_factor")
        sc = D.scenario("your case", fill=fill * f, **kw)
        if kw["work_h_day"] == 14.0:
            sc["days"] = 90
        with out:
            out.clear_output(); show([run_case(scs["S0"]), run_case(sc)])
    btn.on_click(go)
    display(W.VBox(list(w.values()) + [btn, out]))
except ImportError:
    print("ipywidgets is not installed; use the next cell")

In [ ]:
# Without widgets: edit and run
my = D.scenario("my case", fill=fill, robot_bolting=True, tension_speedup=2.0, b6_outfit_factor=0.6)
show([run_case(scs["S0"]), run_case(my)])

## 5. What this lab cannot tell you

- **Real station times, the real takt and the real layout.** All three are estimates. Grounding the model in plant data is the first step the VRB proposes.
- **Effects on safety, ergonomics and quality.** The model counts nacelles, not injuries or defects.
- **Physical feasibility of a robot cell.** That needs the twin's physics episodes, and finally a prototype in the Innovation Lab.

Sources: the twin's report and parameter register (`CuxhavenSiteTwin/report/`, `CuxhavenSiteTwin/site/params.py`).
The model is `CuxhavenSiteTwin/sim/line_des.py`; open it in the file browser.